# Multi-Head Latent Attention

## What is Multi-Head Latent Attention?
Multi-Head Latent Attention operates in a compressed latent space rather than the full feature space, projecting inputs to a lower-dimensional representation before computing attention.

## Why is it Used?
- **Computational Efficiency**: Reduces memory and compute requirements by working in lower dimensions
- **Scalability**: Enables processing of longer sequences with limited resources
- **Parameter Reduction**: Fewer parameters compared to standard multi-head attention

## Key Benefits
- ✅ Significant reduction in computational complexity (O(n²d) → O(n²d_latent))
- ✅ Lower memory footprint
- ✅ Faster inference and training times
- ✅ Maintains competitive performance despite compression
- ✅ Better suited for resource-constrained environments

## Comparison with Other Attention Mechanisms

| Mechanism | Key-Value Sharing | Latent Space | Complexity |
|-----------|-------------------|--------------|------------|
| **Multi-Head Attention** | Independent K,V per head | Full dimension | Highest |
| **Multi-Query Attention** | Single K,V shared across heads | Full dimension | Reduced |
| **Grouped Query Attention** | K,V shared within groups | Full dimension | Moderate |
| **Multi-Head Latent Attention** | Independent per head | Compressed dimension | Lowest |

### Main Differences
- **vs Multi-Head**: Uses latent projection; trades some expressiveness for efficiency
- **vs Multi-Query**: Different efficiency approach; latent focuses on dimension reduction, MQA on parameter sharing
- **vs Grouped Query**: Combines benefits of both MHA and MQA; latent achieves efficiency through compression rather than sharing

1. Initial setup and parameters

- Embedding dimension d_model = 8 
- KV cache dimension d_latent_dim = 4
- Number of heads n_heads = 2
- Head dimension d_head = d_model / n_heads = 8 / 2 = 4
- Input token embedding X = [0.1, -0.2,0.3, 0.4, -0.5,0.6, -0.7,0.8]
- Prior context: 5 tokens ("The next day is bright")


In [ ]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F

class RopelessMLA(nn.Module): ## in advanced version we use RoPE but we will use it later 
    def __init__(self, d_model,n_heads,kv_latent_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.dh = d_model // n_heads ## dimension per head 

        ## projection layers 
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.W_uk = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.W_uv = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias = False)
        self.ln = nn.LayerNorm(kv_latent_dim)
        self.register_buffer('absorbed_k', None) ## Hold W_q, @ W_uk 

    def forward(self,x,kv_cache = None, past_length = 0):
        B,S,D = x.size() # Batch, Seq_len, Dim

        # compute the absorbed_k once : W_q @ W_uk , shape (d_model, latent_dim)
        if self.absorbed_k is None: 
            absorbed = torch.matmul(self.W_q.weight, self.W_uk.weight) # (d_model, latent_dim)
            self.aborbed_k = absorbed.view(self.n_heads, self.dh, -1 ) # (n_heads, d_head, latent_dim )

        ## compress input x to get latent kv representations or space 
        new_c_kv  = self.ln(self.W_dkv(x)) # (B,S, latent_dim)
        if kv_cache is None: 
            c_kv = new_c_kv
        else: 
            c_kv = torch.cat([kv_cache, new_c_kv], dim=1) # (B, S_past + S, latent_dim)

        S_full = c_kv.size(1) # total sequence length including past 
        # decompose V to full d_model and split into heads 

        v_full = self.W_uv(c_kv) # (B, S_full, d_model)
        v = v_full.view(B, S_full, self.n_heads, self.dh).transpose(1, 2) # (B, n_heads, S_full, d_head)
        # use input x directly (since W_q is absorbed) 
        q = x.view(B, S, self.n_heads, self.dh) # (B, S, n_heads, d_head)

        attn_score = torch.zeros(B, self.n_heads, S, S_full, device=x.device) # (B, n_heads, S, S_full)
        for h in range(self.n_heads):
            tmp = torch.matmul(q[:,:,h], self.absorbed_k[h]) # (B, S, latent_dim)
            attn_score[:,h] = torch.bmm(tmp, c_kv.transpose(1,2))  # (B, S, S_full) and 
            
        # scale and apply casual mask
        attn_score = attn_score / (self.dh ** 0.5)
        mask = torch.tril(torch.ones(S,S_full, device  = x.device), diagonal=past_length)
        attn_score = attn_score.masked_fill(mask.view(1,1,S,S_full) == 0, float('-inf'))   
        
        attn_weights = F.softmax(attn_score, dim=-1) # (B, n_heads, S, S_full)

        ## apply the attention weights to  each head's V seperately 
        out_heads = []
        for h in range(self.n_heads):
            context_h = torch.bmm(attn_weights[:,h], v[:,h]) # (B, S, d_head)
            out_heads.append(context_h)

        ## concat heads and project out 
        out = torch.cat(out_heads, dim=-1) # (B, S, d_model)
        return self.W_o(out), c_kv # (B, S, d_model), (B, S_full, latent_dim)
    

# Example usage
d_model = 8
n_heads = 2
kv_latent_dim = 4
mla = RopelessMLA(d_model, n_heads, kv_latent_dim)
x = torch.tensor([[[0.1, -0.2, 0.3, 0.4, -0.5, 0.6, -0.7, 0.8]]])  # (B=1, S=1, D=8)
out, kv_cache = mla(x)
print("Output shape:", out.shape)  # Expected: (1, 1, 8)


